# UK Biobank prompt and classifier validation

This notebook compares six prompt strategies across four instruction-tuned LLMs and two encoder baselines. It uses a fixed, reproducible development/held-out design so prompt selection and encoder-threshold calibration do not use the held-out labels.

**Inputs**

- `ukb_ground_truth_positive_labelled.csv`: known UKB-use publications (`label=1`)
- `ukb_negative_pre2013_labelled_final.csv`: pre-2013 negative baseline (`label=0`)

Both files must contain `id`, `title`, `abstract`, `year`, and `label`.

**Workflow**

1. Reserve reproducible one-shot and five-shot examples outside the benchmark.
2. Sample 4,000 positives and 4,000 negatives using a fixed seed.
3. Split the benchmark into an 80% development set and a 20% held-out test set, stratified by label.
4. Compare all prompts on the development set and select one using a pre-specified mean-LLM F1 rule.
5. Calibrate SciBERT and MiniLM similarity thresholds using development data only.
6. Freeze the selected prompt and thresholds before held-out evaluation.
7. Run all prompt/model combinations on the held-out set for robustness plots; only the pre-selected prompt is used for the primary held-out performance claim.



In [ ]:
%pip -q install --upgrade --no-cache-dir           "transformers==4.48.3" "accelerate==1.3.0"           "bitsandbytes==0.50.2" "sentence-transformers==3.4.1"           "huggingface-hub==0.28.1" "scikit-learn==1.6.1" seaborn

import ast
import gc
import getpass
import hashlib
import importlib.metadata
import json
import os
import platform
import random
import re
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import bitsandbytes as bnb
from google.colab import drive
from huggingface_hub import HfApi, login
from IPython.display import display
from sentence_transformers import SentenceTransformer
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid")
drive.mount("/content/drive")

## Configuration

Edit only the two input paths if your files are stored elsewhere in Drive.

In [ ]:
POSITIVE_PATH = Path(
    "/content/drive/MyDrive/UKB_validation_notebooks/input/"
    "ukb_ground_truth_positive_labelled.csv"
)
NEGATIVE_PATH = Path(
    "/content/drive/MyDrive/UKB_validation_notebooks/input/"
    "ukb_negative_pre2013_labelled_final.csv"
)
OUT_DIR = Path(
    "/content/drive/MyDrive/UKB_validation_notebooks/output/"
    "ukb_prompt_validation_heldout_v3"
)
INTERMEDIATE_DIR = OUT_DIR / "intermediate"
TABLE_DIR = OUT_DIR / "tables"
FIGURE_DIR = OUT_DIR / "figures"
for directory in (OUT_DIR, INTERMEDIATE_DIR, TABLE_DIR, FIGURE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

N_POS = 4_000
N_NEG = 4_000
HELDOUT_FRACTION = 0.20
SAMPLING_SEED = 42
MAX_INPUT_TOKENS = 4_096
MAX_NEW_TOKENS = 32
BATCH_SIZE = 8
MAX_PARSE_ATTEMPTS = 3
USE_4BIT = True

LCDS_PALETTE = [
    "#344874", "#FFBB00", "#5B8DB8",
    "#2A9D8F", "#E76F51", "#8E6C9E",
]

LLM_SPECS = [
    ("Qwen/Qwen2.5-7B-Instruct", "qwen2_5_7b", "Qwen2.5-7B"),
    ("meta-llama/Meta-Llama-3-8B-Instruct", "llama3_8b", "Llama-3-8B"),
    ("mistralai/Mistral-7B-Instruct-v0.3", "mistral_7b", "Mistral-7B"),
    ("HuggingFaceH4/zephyr-7b-beta", "zephyr_7b", "Zephyr-7B"),
]
ENCODER_SPECS = [
    ("allenai/scibert_scivocab_uncased", "scibert_sim", "SciBERT"),
    ("sentence-transformers/all-MiniLM-L6-v2", "sbert_minilm_sim", "MiniLM"),
]
MODEL_TAGS = [x[1] for x in LLM_SPECS + ENCODER_SPECS]
MODEL_LABELS = {x[1]: x[2] for x in LLM_SPECS + ENCODER_SPECS}

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU runtime is required. In Colab select Runtime > Change runtime type > GPU.")

random.seed(SAMPLING_SEED)
np.random.seed(SAMPLING_SEED)
torch.manual_seed(SAMPLING_SEED)
torch.cuda.manual_seed_all(SAMPLING_SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True, warn_only=True)

hf_token = os.getenv("HF_TOKEN", "").strip()
if not hf_token:
    hf_token = getpass.getpass(
        "Hugging Face token (required for gated Llama access): "
    ).strip()
if not hf_token:
    raise ValueError("A Hugging Face token is required for the complete six-model benchmark.")
login(token=hf_token, add_to_git_credential=False)

gpu_name = torch.cuda.get_device_name(0)
COMPUTE_DTYPE = torch.float16

print("GPU:", gpu_name)
print("bitsandbytes:", bnb.__version__)
print("Compute dtype:", COMPUTE_DTYPE)
print("Output directory:", OUT_DIR)

## Input preparation and locked split

The demonstration papers are reserved first and cannot enter either evaluation split. Sorting by ID before seeded sampling makes the selection insensitive to source row order.

In [ ]:
REQUIRED_COLUMNS = ["id", "title", "abstract", "year", "label"]


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def extract_text(value):
    if pd.isna(value):
        return ""
    if not isinstance(value, str):
        return str(value).strip()
    text = value.strip()
    if text.startswith("{") and text.endswith("}"):
        try:
            parsed = ast.literal_eval(text)
            if isinstance(parsed, dict):
                if parsed.get("preferred") is not None:
                    return str(parsed["preferred"]).strip()
                for candidate in parsed.values():
                    if candidate is not None and str(candidate).strip():
                        return str(candidate).strip()
        except Exception:
            pass
    return text


def load_labelled(path, expected_label):
    if not path.exists():
        raise FileNotFoundError(path)
    frame = pd.read_csv(path, low_memory=False)
    missing = sorted(set(REQUIRED_COLUMNS) - set(frame.columns))
    if missing:
        raise ValueError(f"Missing columns in {path.name}: {missing}")
    frame = frame[REQUIRED_COLUMNS].copy()
    frame["id"] = frame["id"].astype(str).str.strip()
    frame["title"] = frame["title"].apply(extract_text)
    frame["abstract"] = frame["abstract"].apply(extract_text)
    frame["year"] = pd.to_numeric(frame["year"], errors="coerce").astype("Int64")
    frame["label"] = int(expected_label)
    frame = frame[frame["id"].ne("")]
    frame = frame[frame["abstract"].str.strip().ne("")]
    frame = frame.drop_duplicates("id", keep="first")
    return frame.sort_values("id").reset_index(drop=True)


positive_pool = load_labelled(POSITIVE_PATH, 1)
negative_pool = load_labelled(NEGATIVE_PATH, 0)

overlap = set(positive_pool["id"]) & set(negative_pool["id"])
if overlap:
    raise ValueError(f"Positive and negative inputs overlap on {len(overlap)} IDs.")

def contains_explicit_ukb(row):
    text = f"{row['title']} {row['abstract']}".lower()
    return any(term in text for term in [
        "uk biobank", "ukb", "ukbb", "united kingdom biobank",
    ])


def contains_hard_negative_cue(row):
    text = f"{row['title']} {row['abstract']}".lower()
    return any(term in text for term in [
        "unlike uk biobank", "compared with uk biobank",
        "such as uk biobank", "including uk biobank",
        "biobanks such as", "ethical", "governance",
        "china kadoorie", "biobank japan", "finngen",
        "all of us", "janus serum bank",
    ])


def reproducible_choice(preferred, fallback, n, seed, excluded=None):
    excluded = set() if excluded is None else set(excluded)
    preferred = preferred[~preferred["id"].isin(excluded)].sort_values("id")
    fallback = fallback[~fallback["id"].isin(excluded)].sort_values("id")
    selected = preferred.sample(n=min(n, len(preferred)), random_state=seed)
    if len(selected) < n:
        needed = n - len(selected)
        remainder = fallback[~fallback["id"].isin(selected["id"])]
        selected = pd.concat([
            selected,
            remainder.sample(n=needed, random_state=seed + 100),
        ])
    return selected.sort_values("id").reset_index(drop=True)


positive_no_explicit = positive_pool[
    ~positive_pool.apply(contains_explicit_ukb, axis=1)
]
negative_hard = negative_pool[
    negative_pool.apply(contains_hard_negative_cue, axis=1)
]

demo_positive = reproducible_choice(
    positive_no_explicit, positive_pool, 3, SAMPLING_SEED + 1
)
demo_negative = reproducible_choice(
    negative_hard, negative_pool, 2, SAMPLING_SEED + 2
)
demonstrations = pd.concat([demo_positive, demo_negative], ignore_index=True)
demonstrations["demo_role"] = [
    "one-shot and five-shot", "five-shot", "five-shot",
    "one-shot and five-shot", "five-shot",
]
demonstrations.to_csv(TABLE_DIR / "few_shot_examples.csv", index=False)

demo_ids = set(demonstrations["id"])
positive_available = positive_pool[~positive_pool["id"].isin(demo_ids)]
negative_available = negative_pool[~negative_pool["id"].isin(demo_ids)]
if len(positive_available) < N_POS or len(negative_available) < N_NEG:
    raise ValueError(
        f"Insufficient records after reserving demonstrations: "
        f"positive={len(positive_available)}, negative={len(negative_available)}"
    )

sampled_positive = positive_available.sample(
    n=N_POS, random_state=SAMPLING_SEED
)
sampled_negative = negative_available.sample(
    n=N_NEG, random_state=SAMPLING_SEED
)
benchmark = pd.concat([sampled_positive, sampled_negative], ignore_index=True)
benchmark = benchmark.sort_values("id").reset_index(drop=True)

development, heldout = train_test_split(
    benchmark,
    test_size=HELDOUT_FRACTION,
    random_state=SAMPLING_SEED,
    stratify=benchmark["label"],
)
development = development.sort_values("id").reset_index(drop=True)
heldout = heldout.sort_values("id").reset_index(drop=True)
development["split"] = "development"
heldout["split"] = "heldout"

development.to_csv(TABLE_DIR / "benchmark_development.csv", index=False)
heldout.to_csv(TABLE_DIR / "benchmark_heldout_test.csv", index=False)
pd.concat([development, heldout], ignore_index=True).to_csv(
    TABLE_DIR / "benchmark_with_split.csv", index=False
)

input_manifest = pd.DataFrame([
    {
        "role": "positive",
        "path": str(POSITIVE_PATH),
        "sha256": sha256_file(POSITIVE_PATH),
        "rows": len(positive_pool),
        "unique_ids": positive_pool["id"].nunique(),
    },
    {
        "role": "negative",
        "path": str(NEGATIVE_PATH),
        "sha256": sha256_file(NEGATIVE_PATH),
        "rows": len(negative_pool),
        "unique_ids": negative_pool["id"].nunique(),
    },
])
input_manifest.to_csv(TABLE_DIR / "input_manifest.csv", index=False)

split_summary = (
    pd.concat([development, heldout])
    .groupby(["split", "label"], as_index=False)
    .size()
)
print("Positive pool:", len(positive_pool))
print("Negative pool:", len(negative_pool))
display(demonstrations[["id", "year", "label", "demo_role"]])
display(split_summary)

## Prompt definitions

The six prompt conditions are retained verbatim in the saved notebook. Every LLM receives the same paper text and decoding budget within a condition.

In [ ]:
one_positive = demo_positive.iloc[0]
one_negative = demo_negative.iloc[0]
five_shot_examples = [
    *demo_positive.assign(label=True).to_dict("records"),
    *demo_negative.assign(label=False).to_dict("records"),
]
random.Random(SAMPLING_SEED).shuffle(five_shot_examples)


def truncate_text(value, limit):
    return str(value or "").strip()[:limit]


def json_instruction():
    return 'Return exactly one JSON object: {"implies_UKB_use": true} or {"implies_UKB_use": false}.'


def make_prompt_v1_conservative(title, abstract):
    return f"""You will be given a scientific paper title and abstract.

Task: decide whether the paper used UK Biobank data or resources.

Definition:
- true: the study used UK Biobank data, participants, samples, imaging, genetics, linked health records, or another UK Biobank resource.
- false: the paper only mentions UK Biobank, discusses biobanks generally, compares with UK Biobank, or uses other biobanks but not UK Biobank.

Be conservative. If uncertain, return false.

{json_instruction()}

title: \"\"\"{truncate_text(title, 700)}\"\"\"
abstract: \"\"\"{truncate_text(abstract, 3200)}\"\"\"
JSON:"""


def make_prompt_v2_balanced(title, abstract):
    return f"""You will be given a scientific paper title and abstract.

Task: decide whether the paper likely used UK Biobank data or resources.

Important:
- Some true UK Biobank-use papers do not mention "UK Biobank" in the abstract.
- The paper may still use UK Biobank if the abstract describes a UK population-scale cohort, genetic/imaging/health-record analysis, or data-resource use that is consistent with UK Biobank.
- Do not require explicit words "UK Biobank" if the evidence strongly suggests use.

Return true when the title/abstract provides reasonable evidence that the paper analysed UK Biobank participants, data, samples, imaging, genetics, linked records, or a UK Biobank-derived cohort.
Return false when the paper is only about generic biobanking, ethics/governance, reviews, comparisons, or another named biobank.

{json_instruction()}
title: \"\"\"{truncate_text(title, 700)}\"\"\"
abstract: \"\"\"{truncate_text(abstract, 3200)}\"\"\"
JSON:"""


def make_prompt_v3_evidence_cues(title, abstract):
    return f"""Classify whether this paper uses UK Biobank.

Use the following cues.

Positive evidence can include:
- explicit UK Biobank / UKB / UKBB mention;
- analysis of a very large UK cohort with genetic, imaging, health-record, lifestyle, biomarker, or hospital-linked data;
- phrases such as participants, cohort, baseline assessment, imaging assessment, genotyping, exome sequencing, linked health records, Hospital Episode Statistics, or Townsend deprivation index in a UK population context;
- a study design that clearly analyses participant-level data rather than merely discussing biobanks.

Negative evidence can include:
- generic discussion of biobanks;
- ethics, governance, consent, infrastructure, sample storage, or review articles;
- use of another biobank only;
- mentions like "such as UK Biobank", "unlike UK Biobank", or comparison with UK Biobank.

Prefer true if the abstract strongly looks like an original analysis using UK Biobank-style data, even if UK Biobank is not named in the abstract.
Prefer false if the abstract is generic or only about other biobanks.

{json_instruction()}
title: \"\"\"{truncate_text(title, 700)}\"\"\"
abstract: \"\"\"{truncate_text(abstract, 3200)}\"\"\"
JSON:"""


def make_prompt_v4_context_no_shot(title, abstract):
    return f"""You will classify a paper using only its title and abstract.

Context:
All papers in this evaluation were retrieved because their full text matched at least one UK Biobank-related query. Therefore, UK Biobank may be mentioned only outside the abstract.

Question:
Based on the title and abstract, is it likely that the paper used UK Biobank data/resources in its own analysis?

Label true if likely UK Biobank use.
Label false if the paper likely only mentions UK Biobank, discusses biobanks generally, or uses other non-UKB resources.

Do not be overly strict: if the paper is an original epidemiological, genetic, imaging, biomarker, or clinical-risk study and the abstract strongly suggests use of a UK population-scale linked cohort, return true.

{json_instruction()}
title: \"\"\"{truncate_text(title, 700)}\"\"\"
abstract: \"\"\"{truncate_text(abstract, 3200)}\"\"\"
JSON:"""


def make_prompt_v5_one_shot(title, abstract):
    return f"""You will be given a scientific paper title and abstract.

Task:
Decide whether the paper likely used UK Biobank data/resources in its own analysis.

Context:
All papers in this evaluation were retrieved because their full text matched a UK Biobank-related search. Some true positives may not mention UK Biobank in the abstract.

{json_instruction()}

Example positive:
title: \"\"\"{truncate_text(one_positive['title'], 500)}\"\"\"
abstract: \"\"\"{truncate_text(one_positive['abstract'], 1800)}\"\"\"
answer: {{"implies_UKB_use":true}}

Example negative:
title: \"\"\"{truncate_text(one_negative['title'], 500)}\"\"\"
abstract: \"\"\"{truncate_text(one_negative['abstract'], 1800)}\"\"\"
answer: {{"implies_UKB_use":false}}

Now classify this paper.
title: \"\"\"{truncate_text(title, 700)}\"\"\"
abstract: \"\"\"{truncate_text(abstract, 3200)}\"\"\"
JSON:"""


def make_prompt_v6_five_shot(title, abstract):
    blocks = []
    for index, example in enumerate(five_shot_examples, start=1):
        answer = str(bool(example["label"])).lower()
        blocks.append(f"""Example {index}
title: \"\"\"{truncate_text(example['title'], 450)}\"\"\"
abstract: \"\"\"{truncate_text(example['abstract'], 1300)}\"\"\"
answer: {{"implies_UKB_use":{answer}}}""")
    examples = "\n\n".join(blocks)
    return f"""You will be given a scientific paper title and abstract.

Task:
Decide whether the paper likely used UK Biobank data/resources in its own analysis.

Context:
All papers in this evaluation were retrieved because their full text matched a UK Biobank-related search.
Some true UK Biobank-use papers may not mention UK Biobank in the abstract.
Some false positives mention biobanks, UK cohorts, or other biobanks but do not use UK Biobank.

Return true when the paper likely analysed UK Biobank participants, data, samples, imaging, genetics, linked health records, or other UK Biobank resources.
Return false for generic biobank discussion, ethics/governance, reviews, comparisons, or other-biobank-only papers.

{json_instruction()}

Few-shot examples:
{examples}

Now classify this paper.
title: \"\"\"{truncate_text(title, 700)}\"\"\"
abstract: \"\"\"{truncate_text(abstract, 3200)}\"\"\"
JSON:"""


PROMPT_BUILDERS = {
    "p1_conservative": make_prompt_v1_conservative,
    "p2_balanced": make_prompt_v2_balanced,
    "p3_evidence_cues": make_prompt_v3_evidence_cues,
    "p4_context_no_shot": make_prompt_v4_context_no_shot,
    "p5_real_one_shot": make_prompt_v5_one_shot,
    "p6_real_five_shot": make_prompt_v6_five_shot,
}
PROMPT_LABELS = {
    "p1_conservative": "Conservative",
    "p2_balanced": "Balanced",
    "p3_evidence_cues": "Evidence cues",
    "p4_context_no_shot": "Context, no-shot",
    "p5_real_one_shot": "One-shot",
    "p6_real_five_shot": "Five-shot",
}

prompt_manifest = pd.DataFrame([
    {"prompt": name, "display_name": PROMPT_LABELS[name]}
    for name in PROMPT_BUILDERS
])
prompt_manifest.to_csv(TABLE_DIR / "prompt_manifest.csv", index=False)
display(prompt_manifest)

## Resolve and record model revisions

Each repository's current commit is resolved once, recorded, and then used explicitly for every model and tokenizer load in this run.

In [ ]:
api = HfApi(token=hf_token)
resolved_models = []
for model_id, tag, display_name in LLM_SPECS + ENCODER_SPECS:
    info = api.model_info(model_id)
    resolved_models.append({
        "model_id": model_id,
        "tag": tag,
        "display_name": display_name,
        "revision": info.sha,
    })
resolved_models = pd.DataFrame(resolved_models)
resolved_models.to_csv(TABLE_DIR / "resolved_model_revisions.csv", index=False)
REVISION = dict(zip(resolved_models["tag"], resolved_models["revision"]))

packages = [
    "torch", "transformers", "accelerate", "bitsandbytes",
    "sentence-transformers", "huggingface-hub", "scikit-learn",
    "pandas", "numpy", "matplotlib", "seaborn",
]
environment = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "gpu": gpu_name,
    "cuda": torch.version.cuda,
    "compute_dtype": str(COMPUTE_DTYPE),
    "sampling_seed": SAMPLING_SEED,
    "n_positive": N_POS,
    "n_negative": N_NEG,
    "heldout_fraction": HELDOUT_FRACTION,
    "max_input_tokens": MAX_INPUT_TOKENS,
    "max_new_tokens": MAX_NEW_TOKENS,
    "batch_size": BATCH_SIZE,
    "max_parse_attempts": MAX_PARSE_ATTEMPTS,
    "decoding": "greedy; do_sample=False; num_beams=1",
    "quantization": "4-bit NF4 with double quantization" if USE_4BIT else "none",
}
environment["packages"] = {
    package: importlib.metadata.version(package) for package in packages
}
with open(TABLE_DIR / "run_configuration.json", "w") as handle:
    json.dump(environment, handle, indent=2)

display(resolved_models)

## Shared inference and evaluation functions

In [ ]:
def parse_binary_output(text):
    cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", str(text).strip(), flags=re.I | re.S)
    candidates = [cleaned]
    match = re.search(r"\{.*?\}", cleaned, flags=re.S)
    if match:
        candidates.append(match.group(0))
    for candidate in candidates:
        try:
            parsed = json.loads(candidate)
            value = parsed.get("implies_UKB_use") if isinstance(parsed, dict) else None
            if isinstance(value, bool):
                return value
            if str(value).strip().lower() in {"true", "yes", "1"}:
                return True
            if str(value).strip().lower() in {"false", "no", "0"}:
                return False
        except Exception:
            pass
    direct = re.fullmatch(r"\s*(true|false|yes|no|positive|negative)\s*[.!]?\s*", cleaned, flags=re.I)
    if direct:
        return direct.group(1).lower() in {"true", "yes", "positive"}
    keyed = re.search(r'implies_UKB_use["\s:]+(true|false)', cleaned, flags=re.I)
    return keyed.group(1).lower() == "true" if keyed else None


def model_input(tokenizer, prompt):
    if getattr(tokenizer, "chat_template", None):
        return tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}],
            tokenize=False,
            add_generation_prompt=True,
        )
    return prompt


def load_llm(model_id, tag):
    revision = REVISION[tag]
    tokenizer = AutoTokenizer.from_pretrained(
        model_id, revision=revision, token=hf_token, use_fast=True
    )
    tokenizer.padding_side = "left"
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    quantization_config = None
    if USE_4BIT:
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=COMPUTE_DTYPE,
        )
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        revision=revision,
        token=hf_token,
        device_map="auto",
        torch_dtype=COMPUTE_DTYPE,
        quantization_config=quantization_config,
        low_cpu_mem_usage=True,
    ).eval()
    return tokenizer, model


def cached_prediction_is_valid(path, frame):
    if not path.exists():
        return False
    try:
        cached = pd.read_csv(path, usecols=["id"])
        return cached["id"].astype(str).tolist() == frame["id"].astype(str).tolist()
    except Exception:
        return False


def run_llm_prompt(model, tokenizer, model_tag, prompt_name, prompt_fn, split_name, frame):
    prediction_path = INTERMEDIATE_DIR / f"predictions_{split_name}_{prompt_name}_{model_tag}.csv"
    raw_path = INTERMEDIATE_DIR / f"raw_outputs_{split_name}_{prompt_name}_{model_tag}.csv"
    if cached_prediction_is_valid(prediction_path, frame) and raw_path.exists():
        print("Using cache:", prediction_path.name)
        return pd.read_csv(prediction_path)

    predictions = [None] * len(frame)
    attempts_used = np.zeros(len(frame), dtype=int)
    raw_attempts = [[] for _ in range(len(frame))]
    unresolved = list(range(len(frame)))
    started = time.time()

    for attempt in range(1, MAX_PARSE_ATTEMPTS + 1):
        if not unresolved:
            break
        next_unresolved = []
        for start in tqdm(
            range(0, len(unresolved), BATCH_SIZE),
            desc=f"{split_name}:{prompt_name}:{model_tag}:attempt{attempt}",
        ):
            indices = unresolved[start:start + BATCH_SIZE]
            prompts = []
            for index in indices:
                row = frame.iloc[index]
                prompt = prompt_fn(row["title"], row["abstract"])
                if attempt > 1:
                    prompt += "\nYour previous response was not valid. Return only the requested JSON object."
                prompts.append(model_input(tokenizer, prompt))

            inputs = tokenizer(
                prompts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=MAX_INPUT_TOKENS,
            ).to(model.device)
            with torch.inference_mode():
                generated = model.generate(
                    **inputs,
                    max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=False,
                    num_beams=1,
                    use_cache=True,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )
            new_tokens = generated[:, inputs["input_ids"].shape[1]:]
            outputs = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)

            for index, output in zip(indices, outputs):
                raw_attempts[index].append(output)
                attempts_used[index] = attempt
                parsed = parse_binary_output(output)
                if parsed is None:
                    next_unresolved.append(index)
                else:
                    predictions[index] = bool(parsed)
            del inputs, generated, new_tokens
        unresolved = next_unresolved

    runtime = time.time() - started
    result = frame[["id", "label"]].copy()
    result["prediction"] = pd.array(predictions, dtype="boolean")
    result["parse_ok"] = result["prediction"].notna()
    result["attempts_used"] = attempts_used
    result["runtime_seconds"] = runtime
    result.to_csv(prediction_path, index=False)

    raw = frame[["id", "label"]].copy()
    raw["raw_attempts_json"] = [json.dumps(values) for values in raw_attempts]
    raw.to_csv(raw_path, index=False)
    return result


def run_llm_stage(split_name, frame):
    for model_id, model_tag, display_name in LLM_SPECS:
        print(f"\nLoading {display_name}: {model_id}@{REVISION[model_tag]}")
        tokenizer, model = load_llm(model_id, model_tag)
        for prompt_name, prompt_fn in PROMPT_BUILDERS.items():
            run_llm_prompt(
                model, tokenizer, model_tag, prompt_name,
                prompt_fn, split_name, frame,
            )
        del model, tokenizer
        gc.collect()
        torch.cuda.empty_cache()


def metric_row(true_values, predictions, parse_ok, **metadata):
    truth = np.asarray(true_values, dtype=bool)
    parsed = np.asarray(parse_ok, dtype=bool)
    effective = pd.Series(predictions).fillna(False).astype(bool).to_numpy()
    tn, fp, fn, tp = confusion_matrix(truth, effective, labels=[False, True]).ravel()
    return {
        **metadata,
        "n": len(truth),
        "n_parsed": int(parsed.sum()),
        "parse_rate": float(parsed.mean()),
        "accuracy": accuracy_score(truth, effective),
        "precision": precision_score(truth, effective, zero_division=0),
        "recall": recall_score(truth, effective, zero_division=0),
        "f1": f1_score(truth, effective, zero_division=0),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


def best_f1_threshold(labels, scores):
    precision, recall, thresholds = precision_recall_curve(labels, scores)
    f1 = 2 * precision * recall / np.maximum(precision + recall, 1e-12)
    valid = np.arange(len(thresholds))
    best = sorted(
        valid,
        key=lambda i: (f1[i], precision[i], recall[i], thresholds[i]),
        reverse=True,
    )[0]
    return float(thresholds[best]), float(f1[best])

## Development-stage LLM predictions

This is the only stage used to select the prompt.

In [ ]:
run_llm_stage("development", development)

## Encoder similarities and development-only calibration

The encoder baselines are prompt-independent. Their decision thresholds are selected on development data and then applied unchanged to held-out data.

In [ ]:
def mean_pool(last_hidden, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden.size()).float()
    return (last_hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)


def scibert_scores(frame, split_name):
    path = INTERMEDIATE_DIR / f"scores_{split_name}_scibert_sim.csv"
    if path.exists() and cached_prediction_is_valid(path, frame):
        return pd.read_csv(path)
    model_id, tag, _ = ENCODER_SPECS[0]
    revision = REVISION[tag]
    tokenizer = AutoTokenizer.from_pretrained(model_id, revision=revision)
    model = AutoModel.from_pretrained(
        model_id, revision=revision, torch_dtype=COMPUTE_DTYPE
    ).to("cuda").eval()
    query = "This scientific paper uses UK Biobank data or resources."

    def encode(texts, batch_size=64):
        outputs = []
        for start in tqdm(range(0, len(texts), batch_size), desc=f"SciBERT:{split_name}"):
            batch = tokenizer(
                texts[start:start + batch_size], return_tensors="pt",
                padding=True, truncation=True, max_length=512,
            ).to("cuda")
            with torch.inference_mode():
                hidden = model(**batch).last_hidden_state
                pooled = mean_pool(hidden, batch["attention_mask"])
                pooled = torch.nn.functional.normalize(pooled, dim=1)
            outputs.append(pooled.float().cpu().numpy())
        return np.vstack(outputs)

    query_embedding = encode([query])[0]
    texts = (frame["title"] + "\n" + frame["abstract"]).tolist()
    embeddings = encode(texts)
    result = frame[["id", "label"]].copy()
    result["score"] = embeddings @ query_embedding
    result.to_csv(path, index=False)
    del model, tokenizer, embeddings
    gc.collect()
    torch.cuda.empty_cache()
    return result


def minilm_scores(frame, split_name):
    path = INTERMEDIATE_DIR / f"scores_{split_name}_sbert_minilm_sim.csv"
    if path.exists() and cached_prediction_is_valid(path, frame):
        return pd.read_csv(path)
    model_id, tag, _ = ENCODER_SPECS[1]
    model = SentenceTransformer(
        model_id, revision=REVISION[tag], device="cuda"
    )
    query = "This scientific paper uses UK Biobank data or resources."
    texts = (frame["title"] + "\n" + frame["abstract"]).tolist()
    query_embedding = model.encode(
        [query], normalize_embeddings=True, convert_to_numpy=True
    )[0]
    embeddings = model.encode(
        texts, normalize_embeddings=True, convert_to_numpy=True,
        batch_size=128, show_progress_bar=True,
    )
    result = frame[["id", "label"]].copy()
    result["score"] = embeddings @ query_embedding
    result.to_csv(path, index=False)
    del model, embeddings
    gc.collect()
    torch.cuda.empty_cache()
    return result


dev_encoder_scores = {
    "scibert_sim": scibert_scores(development, "development"),
    "sbert_minilm_sim": minilm_scores(development, "development"),
}
threshold_rows = []
for tag, score_frame in dev_encoder_scores.items():
    threshold, development_f1 = best_f1_threshold(
        score_frame["label"].astype(bool).to_numpy(),
        score_frame["score"].to_numpy(),
    )
    threshold_rows.append({
        "model": tag,
        "threshold": threshold,
        "development_f1_at_threshold": development_f1,
    })
encoder_thresholds = pd.DataFrame(threshold_rows)
encoder_thresholds.to_csv(TABLE_DIR / "encoder_thresholds_development.csv", index=False)
THRESHOLD = dict(zip(encoder_thresholds["model"], encoder_thresholds["threshold"]))
display(encoder_thresholds)

## Development metrics and prompt lock

In [ ]:
def read_llm_prediction(split_name, prompt_name, model_tag):
    path = INTERMEDIATE_DIR / f"predictions_{split_name}_{prompt_name}_{model_tag}.csv"
    result = pd.read_csv(path)
    result["prediction"] = result["prediction"].map(
        {True: True, False: False, "True": True, "False": False}
    )
    result["parse_ok"] = result["parse_ok"].astype(str).str.lower().eq("true")
    return result


def collect_metrics(split_name, frame, encoder_scores):
    rows = []
    for prompt_name in PROMPT_BUILDERS:
        for _, tag, _ in LLM_SPECS:
            prediction = read_llm_prediction(split_name, prompt_name, tag)
            rows.append(metric_row(
                prediction["label"], prediction["prediction"], prediction["parse_ok"],
                split=split_name, prompt=prompt_name, model=tag, model_type="LLM",
            ))
        for tag, score_frame in encoder_scores.items():
            prediction = score_frame["score"].ge(THRESHOLD[tag])
            rows.append(metric_row(
                score_frame["label"], prediction, np.ones(len(score_frame), dtype=bool),
                split=split_name, prompt=prompt_name, model=tag,
                model_type="encoder baseline",
            ))
    return pd.DataFrame(rows)


development_metrics = collect_metrics(
    "development", development, dev_encoder_scores
)
development_metrics.to_csv(TABLE_DIR / "development_metrics_all_prompts_models.csv", index=False)

prompt_selection = (
    development_metrics[development_metrics["model_type"].eq("LLM")]
    .groupby("prompt", as_index=False)
    .agg(
        mean_llm_f1=("f1", "mean"),
        mean_llm_precision=("precision", "mean"),
        mean_llm_recall=("recall", "mean"),
        mean_llm_accuracy=("accuracy", "mean"),
        minimum_parse_rate=("parse_rate", "min"),
    )
    .sort_values(
        ["mean_llm_f1", "mean_llm_precision", "mean_llm_recall", "prompt"],
        ascending=[False, False, False, True],
    )
    .reset_index(drop=True)
)
SELECTED_PROMPT = prompt_selection.loc[0, "prompt"]
prompt_selection["selected"] = prompt_selection["prompt"].eq(SELECTED_PROMPT)
prompt_selection.to_csv(TABLE_DIR / "development_prompt_selection.csv", index=False)

prompt_lock = {
    "selected_prompt": SELECTED_PROMPT,
    "selection_rule": (
        "Highest mean development F1 across the four instruction-tuned LLMs; "
        "ties broken by mean precision, mean recall, then prompt name."
    ),
    "sampling_seed": SAMPLING_SEED,
    "encoder_thresholds": THRESHOLD,
}
with open(TABLE_DIR / "selected_prompt_before_heldout.json", "w") as handle:
    json.dump(prompt_lock, handle, indent=2)

print("Selected prompt before held-out evaluation:", SELECTED_PROMPT)
display(prompt_selection)

## Held-out LLM predictions

The prompt choice and encoder thresholds have already been written to disk before this cell runs.

In [ ]:
run_llm_stage("heldout", heldout)

## Held-out encoder scores and final performance tables

In [ ]:
heldout_encoder_scores = {
    "scibert_sim": scibert_scores(heldout, "heldout"),
    "sbert_minilm_sim": minilm_scores(heldout, "heldout"),
}
heldout_metrics = collect_metrics("heldout", heldout, heldout_encoder_scores)
heldout_metrics.to_csv(TABLE_DIR / "heldout_metrics_all_prompts_models.csv", index=False)

primary_heldout_metrics = heldout_metrics[
    heldout_metrics["prompt"].eq(SELECTED_PROMPT)
].copy()
primary_heldout_metrics["model_display"] = primary_heldout_metrics["model"].map(MODEL_LABELS)
primary_heldout_metrics.to_csv(
    TABLE_DIR / "heldout_metrics_selected_prompt_primary.csv", index=False
)

development_metrics["model_display"] = development_metrics["model"].map(MODEL_LABELS)
heldout_metrics["model_display"] = heldout_metrics["model"].map(MODEL_LABELS)
all_metrics = pd.concat([development_metrics, heldout_metrics], ignore_index=True)
all_metrics.to_csv(TABLE_DIR / "all_development_and_heldout_metrics.csv", index=False)

print("Primary held-out performance for pre-selected prompt:", SELECTED_PROMPT)
display(primary_heldout_metrics[[
    "model_display", "n", "parse_rate", "accuracy",
    "precision", "recall", "f1", "tn", "fp", "fn", "tp",
]])

## Combined predictions and pairwise agreement

Pairwise agreement is calculated only on rows parsed by both models. The accompanying `n` tables make the denominator explicit.

In [ ]:
def combined_predictions(split_name, frame, prompt_name, encoder_scores):
    combined = frame[["id", "title", "abstract", "year", "label"]].copy()
    combined["True_label"] = combined["label"].astype(bool)
    for _, tag, _ in LLM_SPECS:
        prediction = read_llm_prediction(split_name, prompt_name, tag)
        combined[tag] = prediction["prediction"]
        combined[f"{tag}_parse_ok"] = prediction["parse_ok"]
    for tag, score_frame in encoder_scores.items():
        combined[tag] = score_frame["score"].ge(THRESHOLD[tag]).to_numpy()
        combined[f"{tag}_parse_ok"] = True
    return combined


def pairwise_agreement(frame):
    agreement = pd.DataFrame(index=MODEL_TAGS, columns=MODEL_TAGS, dtype=float)
    compared = pd.DataFrame(index=MODEL_TAGS, columns=MODEL_TAGS, dtype=int)
    for model_a in MODEL_TAGS:
        for model_b in MODEL_TAGS:
            valid = (
                frame[f"{model_a}_parse_ok"].astype(bool)
                & frame[f"{model_b}_parse_ok"].astype(bool)
            )
            compared.loc[model_a, model_b] = int(valid.sum())
            agreement.loc[model_a, model_b] = (
                frame.loc[valid, model_a].astype(bool)
                .eq(frame.loc[valid, model_b].astype(bool))
                .mean()
            )
    return agreement, compared


pairwise_tables = {}
for prompt_name in PROMPT_BUILDERS:
    combined = combined_predictions(
        "heldout", heldout, prompt_name, heldout_encoder_scores
    )
    combined.to_csv(TABLE_DIR / f"predictions_heldout_{prompt_name}.csv", index=False)
    combined.to_csv(TABLE_DIR / f"predictions_{prompt_name}.csv", index=False)
    agreement, compared = pairwise_agreement(combined)
    agreement_percent = agreement * 100
    agreement_percent.to_csv(
        TABLE_DIR / f"pairwise_agreement_percent_{prompt_name}.csv"
    )
    compared.to_csv(TABLE_DIR / f"pairwise_agreement_n_{prompt_name}.csv")
    pairwise_tables[prompt_name] = agreement

display(pairwise_tables[SELECTED_PROMPT] * 100)

## Pairwise-agreement figures

In [ ]:
agreement_cmap = sns.light_palette(LCDS_PALETTE[0], as_cmap=True)

for prompt_name, agreement in pairwise_tables.items():
    figure, axis = plt.subplots(figsize=(8.2, 7.1))
    sns.heatmap(
        agreement.rename(index=MODEL_LABELS, columns=MODEL_LABELS),
        vmin=0, vmax=1, cmap=agreement_cmap, annot=True, fmt=".2f",
        square=True, cbar_kws={"label": "Pairwise agreement"}, ax=axis,
    )
    axis.set_title(f"Held-out pairwise agreement: {PROMPT_LABELS[prompt_name]}")
    figure.tight_layout()
    figure.savefig(FIGURE_DIR / f"pairwise_agreement_{prompt_name}.png", dpi=300, bbox_inches="tight")
    figure.savefig(FIGURE_DIR / f"pairwise_agreement_{prompt_name}.pdf", bbox_inches="tight")
    plt.close(figure)

figure, axes = plt.subplots(2, 3, figsize=(21, 13))
axes = axes.ravel()
panel_tags = ["a.", "b.", "c.", "d.", "e.", "f."]
for axis, panel_tag, (prompt_name, agreement) in zip(
    axes, panel_tags, pairwise_tables.items()
):
    labelled = agreement.rename(index=MODEL_LABELS, columns=MODEL_LABELS)
    sns.heatmap(
        labelled, vmin=0, vmax=1, cmap=agreement_cmap,
        annot=True, fmt=".2f", square=True, cbar=False, ax=axis,
        annot_kws={"fontsize": 7.5},
    )
    axis.set_title(PROMPT_LABELS[prompt_name], fontsize=12)
    axis.text(-0.18, 1.08, panel_tag, transform=axis.transAxes, fontsize=15, fontweight="bold")
    axis.tick_params(labelsize=8)
colour_axis = figure.add_axes([0.92, 0.20, 0.015, 0.62])
scalar = plt.cm.ScalarMappable(norm=plt.Normalize(0, 1), cmap=agreement_cmap)
figure.colorbar(scalar, cax=colour_axis, label="Pairwise agreement")
figure.suptitle("Pairwise agreement across prompt strategies on the held-out test set", fontsize=16)
figure.subplots_adjust(left=0.08, right=0.90, top=0.93, bottom=0.07, wspace=0.40, hspace=0.42)
figure.savefig(FIGURE_DIR / "pairwise_agreement_all_six_prompts.png", dpi=300, bbox_inches="tight")
figure.savefig(FIGURE_DIR / "pairwise_agreement_all_six_prompts.pdf", bbox_inches="tight")
plt.show()

## Held-out precision, recall and F1 figures

In [ ]:
figure, axes = plt.subplots(1, 3, figsize=(20, 6.5))
for axis, metric in zip(axes, ["precision", "recall", "f1"]):
    table = heldout_metrics.pivot(index="prompt", columns="model", values=metric)
    table = table.reindex(index=list(PROMPT_BUILDERS), columns=MODEL_TAGS)
    table = table.rename(index=PROMPT_LABELS, columns=MODEL_LABELS)
    sns.heatmap(
        table, vmin=0, vmax=1, cmap=agreement_cmap,
        annot=True, fmt=".2f", cbar=metric == "f1", ax=axis,
        cbar_kws={"label": "Score"},
    )
    axis.set_title(metric.capitalize())
    axis.set_xlabel("")
    axis.set_ylabel("")
    axis.tick_params(axis="x", rotation=45, labelsize=8)
    axis.tick_params(axis="y", labelsize=8)
figure.suptitle("Held-out performance across prompts and classifiers", fontsize=16)
figure.tight_layout()
figure.savefig(FIGURE_DIR / "heldout_precision_recall_f1_heatmaps.png", dpi=300, bbox_inches="tight")
figure.savefig(FIGURE_DIR / "heldout_precision_recall_f1_heatmaps.pdf", bbox_inches="tight")
plt.show()

plot_frame = primary_heldout_metrics.melt(
    id_vars=["model_display"], value_vars=["precision", "recall", "f1"],
    var_name="metric", value_name="score",
)
figure, axis = plt.subplots(figsize=(11, 5.8))
sns.barplot(
    data=plot_frame, x="model_display", y="score", hue="metric",
    palette=LCDS_PALETTE[:3], ax=axis,
)
axis.set_ylim(0, 1)
axis.set_xlabel("")
axis.set_ylabel("Held-out score")
axis.set_title(f"Primary held-out performance: {PROMPT_LABELS[SELECTED_PROMPT]}")
axis.tick_params(axis="x", rotation=25)
axis.legend(title="Metric")
figure.tight_layout()
figure.savefig(FIGURE_DIR / "heldout_selected_prompt_performance.png", dpi=300, bbox_inches="tight")
figure.savefig(FIGURE_DIR / "heldout_selected_prompt_performance.pdf", bbox_inches="tight")
plt.show()

## Confusion matrices for six prompts and six classifiers

Each panel reports counts and row-normalised percentages. Unparsed LLM responses are treated as negative, matching the conservative deployment rule, while parsing coverage is reported separately in the metric tables.

In [ ]:
confusion_rows = []
confusion_lookup = {}
for prompt_name in PROMPT_BUILDERS:
    combined = pd.read_csv(TABLE_DIR / f"predictions_heldout_{prompt_name}.csv")
    truth = combined["label"].astype(bool).to_numpy()
    for model_tag in MODEL_TAGS:
        effective = combined[model_tag].map(
            {True: True, False: False, "True": True, "False": False}
        ).fillna(False).astype(bool).to_numpy()
        matrix = confusion_matrix(truth, effective, labels=[False, True])
        confusion_lookup[(prompt_name, model_tag)] = matrix
        tn, fp, fn, tp = matrix.ravel()
        confusion_rows.append({
            "prompt": prompt_name, "model": model_tag,
            "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        })
confusion_counts = pd.DataFrame(confusion_rows)
confusion_counts.to_csv(TABLE_DIR / "heldout_confusion_counts_all_prompts_models.csv", index=False)


def draw_confusion(axis, matrix, title, show_y=True):
    normalised = matrix / np.maximum(matrix.sum(axis=1, keepdims=True), 1)
    axis.imshow(normalised, cmap=agreement_cmap, vmin=0, vmax=1)
    for row in range(2):
        for column in range(2):
            axis.text(
                column, row,
                f"{matrix[row, column]:,}\n({normalised[row, column]:.1%})",
                ha="center", va="center", fontsize=7.5,
            )
    axis.set_xticks([0, 1], ["Pred. negative", "Pred. positive"], fontsize=7)
    axis.set_yticks([0, 1], ["True negative", "True positive"] if show_y else ["", ""], fontsize=7)
    axis.set_title(title, fontsize=9)


figure, axes = plt.subplots(6, 6, figsize=(24, 23))
for row, prompt_name in enumerate(PROMPT_BUILDERS):
    for column, model_tag in enumerate(MODEL_TAGS):
        draw_confusion(
            axes[row, column],
            confusion_lookup[(prompt_name, model_tag)],
            MODEL_LABELS[model_tag] if row == 0 else "",
            show_y=column == 0,
        )
        if column == 0:
            axes[row, column].set_ylabel(PROMPT_LABELS[prompt_name], fontsize=10, fontweight="bold")
figure.suptitle("Held-out confusion matrices: six prompts by six classifiers", fontsize=18)
figure.tight_layout(rect=[0, 0, 1, 0.98])
figure.savefig(FIGURE_DIR / "heldout_confusion_matrices_6_prompts_6_models.png", dpi=300, bbox_inches="tight")
figure.savefig(FIGURE_DIR / "heldout_confusion_matrices_6_prompts_6_models.pdf", bbox_inches="tight")
plt.show()

for prompt_name in PROMPT_BUILDERS:
    figure, axes = plt.subplots(2, 3, figsize=(13, 9))
    for axis, model_tag in zip(axes.ravel(), MODEL_TAGS):
        draw_confusion(
            axis, confusion_lookup[(prompt_name, model_tag)],
            MODEL_LABELS[model_tag], show_y=True,
        )
    figure.suptitle(f"Held-out confusion matrices: {PROMPT_LABELS[prompt_name]}", fontsize=15)
    figure.tight_layout(rect=[0, 0, 1, 0.96])
    figure.savefig(FIGURE_DIR / f"heldout_confusion_matrices_{prompt_name}.png", dpi=300, bbox_inches="tight")
    figure.savefig(FIGURE_DIR / f"heldout_confusion_matrices_{prompt_name}.pdf", bbox_inches="tight")
    plt.close(figure)

display(confusion_counts.head(12))

## Output inventory

In [ ]:
inventory = []
for path in sorted(OUT_DIR.rglob("*")):
    if path.is_file():
        inventory.append({
            "relative_path": str(path.relative_to(OUT_DIR)),
            "size_bytes": path.stat().st_size,
        })
inventory = pd.DataFrame(inventory)
inventory.to_csv(OUT_DIR / "output_inventory.csv", index=False)
print("Selected prompt:", SELECTED_PROMPT)
print("Output files:", len(inventory))
print("Primary held-out metrics:", TABLE_DIR / "heldout_metrics_selected_prompt_primary.csv")
print("Combined agreement figure:", FIGURE_DIR / "pairwise_agreement_all_six_prompts.png")
print("Combined confusion figure:", FIGURE_DIR / "heldout_confusion_matrices_6_prompts_6_models.png")
display(inventory.tail(30))